In [1]:
# extracted data from lidl plus receipt
receipt = [
    "Banane Fair Bio",
    "Zwiebeln weiß",
    "Bioland Zucchini",
    "Petersilie",
    "Bio Limetten",
    "Weidemilch 3,8%",
    "Exquisa fitline Natur",
    "Bio-Eier OKT 10er",
    "Bio Müsli Schoko",
    "Lava Bar Cookie",
    "Dinkelschnitte",
    "Walnusskerne",
    "Bio Cashewkerne",
    "Rinderhackfleisch 5%",
    "Puten-Geschnetzeltes"
]

In [2]:
!pip install requests pandas

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached pandas-3.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached charset_normalizer-3.4.7-cp311-cp311-macosx_10_9_universal2.whl.metadata (40 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.4.7-cp311-cp311-macosx_10_9_universal2.whl (309 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Using cached pandas-3.0.3-cp311-cp311-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_arm64.whl (5.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pandas]━━━━ 4/5 [pandas]_normalizer]


In [ ]:
import requests
import pandas as pd
from time import sleep

BASE_URL = "https://world.openfoodfacts.org/cgi/search.pl"

headers = {
    "User-Agent": "NutritionOptimizerPrototype/0.1 (student project)"
}

def search_product(product_name):

    params = {
        "search_terms": product_name,
        "search_simple": 1,
        "action": "process",
        "json": 1,
        "page_size": 3
    }

    response = requests.get(
        BASE_URL,
        params=params,
        headers=headers
    )

    data = response.json()

    return data.get("products", [])

for item in receipt:

    print("="*60)
    print(item)

    products = search_product(item)

    print(f"{len(products)} matches")

    for p in products:

        print(
            p.get("product_name"),
            "|",
            p.get("brands"),
            "|",
            p.get("countries")
        )

In [9]:
def search_product(product_name):

    params = {
        "search_terms": product_name,
        "search_simple": 1,
        "action": "process",
        "json": 1,
        "page_size": 3,
        "fields": "product_name,brands,countries,nutriscore_grade"
    }

    response = requests.get(BASE_URL, params=params, headers=headers, timeout=10)
    response.raise_for_status()

    data = response.json()

    results = []

    for p in data.get("products", []):
        results.append({
            "query": product_name,
            "product_name": p.get("product_name"),
            "brands": p.get("brands"),
            "countries": p.get("countries"),
            "nutriscore": p.get("nutriscore_grade")
        })

    return results

In [13]:
headers = {
    "User-Agent": "NutritionOptimizerPrototype/0.1 (student project)",
    "Accept": "application/json"
}

search_product("Petersilie")

[{'query': 'Petersilie',
  'product_name': 'Pesto Basilico e Rucula',
  'brands': 'Barilla',
  'countries': 'Australia, Austria, Belgium, France, Germany, Italy, Liechtenstein, Réunion, Romania, Switzerland',
  'nutriscore': 'e'},
 {'query': 'Petersilie',
  'product_name': 'Petersilie getrocknet',
  'brands': 'Kania',
  'countries': 'France, Germany',
  'nutriscore': 'a'},
 {'query': 'Petersilie',
  'product_name': 'Petersilie - Erntefrisch, Gefriergetrocknet',
  'brands': 'Gut & Günstig',
  'countries': 'Germany',
  'nutriscore': 'a'}]

In [ ]:
# add "germany" as country to better match products

In [14]:
import requests

def get_product_by_ean(ean: str):
    url = f"https://world.openfoodfacts.org/api/v0/product/{ean}.json"
    
    r = requests.get(url)
    data = r.json()
    
    if data.get("status") != 1:
        return None  # product not found
    
    product = data["product"]
    
    return {
        "barcode": product.get("code"),
        "name": product.get("product_name"),
        "brand": product.get("brands"),
        "ingredients": product.get("ingredients_text"),
        "nutriscore": product.get("nutriscore_grade"),
        "nova_group": product.get("nova_group"),
        "categories": product.get("categories")
    }


# example
print(get_product_by_ean("9920758"))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)